# 🛡️ Rapport de Démonstration : Security Log Stream Engine
## Projet Python ING3 - ECE Paris

**Auteurs :** Fatima Medaouri, Sidi Mohamed, Hafsa  
**Sujet :** Traitement de flux massifs de logs en streaming (jusqu'à 10-15 Go sans saturation mémoire), détection d'intrusions et audit de sécurité.

---
### Sommaire
1. **Architecture & Pipeline de données** (Streaming avec `yield`, `filter`, `map`)
2. **Exécution du Pipeline & Analyse en temps réel**
3. **Visualisation Graphique du Trafic Réseau**
4. **Visualisation Graphique des Alertes de Sécurité Détectées**
5. **Analyse Mathématique d'Ensembles (`set`, `frozenset`)**

## 1. Importation des modules et configuration
Tous les modules développés dans le cadre du projet sont importés depuis le package `src`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

# Modules du projet
from src.ingestion import ingestion_multi_sources
from src.pipeline import filtrer_les_logs, normaliser_flux
from src.analyzer import agreger_statistiques, charger_blacklist
from src.detector import detecter_force_brute
from src.rules_engine import apply_security_rules, blacklisted_ip, web_injection, critical_level
from collections import Counter

log_files = sorted(Path("data/sample").glob("*.log"))
blacklist = charger_blacklist("data/sample/blacklist.txt")

print(f"Fichiers détectés : {[f.name for f in log_files]}")
print(f"Blacklist chargée (frozenset) : {len(blacklist)} adresses IP surveillées")

## 2. Exécution du Pipeline de Streaming
Le pipeline lit les logs en flux tendu sans charger les fichiers entiers en mémoire vive.

In [ ]:
def get_stream():
    return normaliser_flux(filtrer_les_logs(ingestion_multi_sources(log_files)))

# 1. Règles de sécurité
alerts = list(apply_security_rules(blacklisted_ip, web_injection, critical_level, events=get_stream(), blacklist=blacklist))
alert_counts = Counter(a.rule for a in alerts)

# 2. Détection de Force Brute
brute_force_alerts = detecter_force_brute(get_stream(), seuil_echecs=3)
if brute_force_alerts:
    alert_counts["force_brute"] = len(brute_force_alerts)

# 3. Statistiques & Ensembles mathématiques
stats = agreger_statistiques(get_stream(), blacklist)

print("=== RÉSUMÉ DE L'ANALYSE ===")
print(f"Total alertes règles : {len(alerts)}")
print(f"Alertes force brute : {len(brute_force_alerts)}")
print(f"IP uniques actives : {len(stats['ips_actives'])}")
print(f"IP actives suspectes (Intersection &) : {len(stats['operations_ensembles']['hostiles'])}")

## 3. Visualisation Graphique : Trafic par Sévérité
Ce graphique illustre le volume d'événements selon leur gravité (`INFO`, `WARNING`, `ERROR`, `CRITICAL`), collectés de manière dynamique avec `defaultdict(int)` sans risque de `KeyError`.

In [ ]:
niveaux = stats["total_par_niveau"]
labels = list(niveaux.keys())
valeurs = list(niveaux.values())
couleurs = ["#2ecc71" if l == "INFO" else "#f39c12" if l == "WARNING" else "#e74c3c" if l == "ERROR" else "#8e44ad" for l in labels]

plt.figure(figsize=(9, 5))
bars = plt.bar(labels, valeurs, color=couleurs, edgecolor="black", alpha=0.85)
plt.title("Volume du Trafic Réseau par Niveau de Sévérité", fontsize=14, fontweight="bold")
plt.xlabel("Niveau de log", fontsize=12)
plt.ylabel("Nombre d'événements", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Ajout des valeurs au-dessus des barres
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 100, f"{yval:,}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

## 4. Visualisation Graphique : Répartition des Menaces Détectées
Répartition des menaces identifiées par le moteur de règles et l'analyse de force brute.

In [ ]:
labels_menaces = list(alert_counts.keys())
valeurs_menaces = list(alert_counts.values())
colors = ["#e74c3c", "#d35400", "#c0392b", "#962d3e"]

plt.figure(figsize=(8, 6))
plt.pie(
    valeurs_menaces, 
    labels=labels_menaces, 
    autopct="%1.1f%%", 
    startangle=140, 
    colors=colors[:len(labels_menaces)],
    wedgeprops={"edgecolor": "black", "linewidth": 1.2}
)
plt.title("Répartition des Alertes de Sécurité Détectées", fontsize=14, fontweight="bold")
plt.show()

## 5. Synthèse Mathématique des Ensembles d'IP
Validation des 4 opérations ensemblistes :

In [ ]:
op = stats["operations_ensembles"]
print(f"1. Union (|)               : {len(op['union'])} adresses uniques totales")
print(f"2. Intersection (&)        : {len(op['hostiles'])} adresses suspectes actives")
print(f"3. Différence (-)          : {len(op['saines'])} adresses saines actives")
print(f"4. Différence symétrique (^): {len(op['divergentes'])} adresses exclusives")
print("\nAdresses hostiles détectées dans le flux :", op['hostiles'])